# Description
We demonstrate how to extract a concept vector for a concept using the method outlined by Lindsey et. al. in their ["Emergent Introspective Awareness"](https://transformer-circuits.pub/2025/introspection/index.html) paper. This is for Llama-8.1-3B-Instruct, but should work with minor modifications for any model in the Llama-3.1 family.

We define a few global configuration variables, note that you'll need a `.env` file with the values that we retrieve from the environment in the below cell.

In [1]:
from dotenv import load_dotenv
import os

MODEL = "meta-llama/Llama-3.3-70B-Instruct"
NDIF_API_KEY = os.environ["NDIF_API_KEY"]

Begin with a health check on the NDIF API, which is used throughout this notebook.

In [2]:
import nnsight
from pprint import pprint

assert nnsight.is_model_running(MODEL), f"{MODEL} is not online."

status = nnsight.ndif_status()

print(status)

target = MODEL

NDIF Service: Up 🟢 

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Model Class   ┃ Repo ID                            ┃ Revision ┃ Type      ┃ Status       ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ LanguageModel │ meta-llama/Llama-3.1-70B-Instruct  │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ google/gemma-2-9b-it               │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ openai-community/gpt2              │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-405B          │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.3-70B-Instruct  │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-70B           │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-8B            │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ EleutherAI/gpt-j-6b                │ main     │ Dedicated │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-405B-Instruct │ main     │ Scheduled │ NOT DEPLOYED │
└───────────────┴────────────────────────────────────┴──────────┴───────────┴──────────────┘

Load the model and perform a quick test.

In [3]:
import nnsight

model = nnsight.LanguageModel(MODEL)

prompt = "Reply with exactly three words:"

with model.generate(prompt, max_new_tokens=10, do_sample=False, remote=True):
    out_ids = model.generator.output.save()

print(out_ids)

text = model.tokenizer.decode(out_ids, skip_special_tokens=True)
print(text)

⬇ Downloading:   0%|          | 0.00/672 [00:00<?]

tensor([[128000,  21509,    449,   7041,   2380,   4339,     25,    358,   1097,
           6380,    627,     40,   1097,   6380,     13,    720,     40]])
['Reply with exactly three words: I am happy.\nI am happy. \nI']


Test if we can extract the activations of the model at each layer.

In [4]:
word = "recursion"
messages = [{"role": "user", "content": f"Tell me about {word}."}]
prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print(prompt)

# demonstration of how to extract activations from each layer    
num_layers = model.config.num_hidden_layers
with model.trace(prompt, remote=True) as tracer:
    hidden_states = []
    for i in range(num_layers):
        hidden_states.append(model.model.layers[i].output[0].detach().cpu().save())
    hidden_states.save()

# Access after the trace
print(hidden_states[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell me about recursion.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




⬇ Downloading:   0%|          | 0.00/40.8M [00:00<?]

tensor([[ 0.0019,  0.0052,  0.0018,  ...,  0.0040,  0.0053,  0.0068],
        [ 0.0019,  0.0052,  0.0018,  ...,  0.0040,  0.0053,  0.0068],
        [ 0.0003, -0.0059,  0.0004,  ...,  0.0101, -0.0152, -0.0043],
        ...,
        [ 0.0037,  0.0281, -0.0121,  ..., -0.0019,  0.0002,  0.0057],
        [-0.0025, -0.0002,  0.0004,  ...,  0.0051, -0.0016, -0.0028],
        [ 0.0068,  0.0016,  0.0021,  ...,  0.0011, -0.0012,  0.0015]],
       dtype=torch.bfloat16)


Compute mean baseline last token activation vector using the 100 words in Lindsey et al.

In [5]:
import torch

# Lindsey et al. baseline words
words = "Desks, Jackets, Gondolas, Laughter, Intelligence, Bicycles, Chairs, Orchestras, Sand, Pottery, Arrowheads, Jewelry, Daffodils, Plateaus, Estuaries, Quilts, Moments, Bamboo, Ravines, Archives, Hieroglyphs, Stars, Clay, Fossils, Wildlife, Flour, Traffic, Bubbles, Honey, Geodes, Magnets, Ribbons, Zigzags, Puzzles, Tornadoes, Anthills, Galaxies, Poverty, Diamonds, Universes, Vinegar, Nebulae, Knowledge, Marble, Fog, Rivers, Scrolls, Silhouettes, Marbles, Cakes, Valleys, Whispers, Pendulums, Towers, Tables, Glaciers, Whirlpools, Jungles, Wool, Anger, Ramparts, Flowers, Research, Hammers, Clouds, Justice, Dogs, Butterflies, Needles, Fortresses, Bonfires, Skyscrapers, Caravans, Patience, Bacon, Velocities, Smoke, Electricity, Sunsets, Anchors, Parchments, Courage, Statues, Oxygen, Time, Butterflies, Fabric, Pasta, Snowflakes, Mountains, Echoes, Pianos, Sanctuaries, Abysses, Air, Dewdrops, Gardens, Literature, Rice, Enigmas"
baseline_words = [w.strip().lower() for w in words.split(",") if w.strip()]

num_layers = model.config.num_hidden_layers
baseline_samples = []

NUM_WORDS = 5

# you could test it with the first five samples to reduce computation time
for w in baseline_words[:NUM_WORDS]:
    messages = [{"role": "user", "content": f"Tell me about {w}"}]
    prompt = model.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    with model.trace(prompt, remote=True) as tracer:
        hidden_states = []
        for i in range(num_layers):
            hidden_states.append(model.model.layers[i].output[0][-1, :].detach().cpu().save())
        hidden_states.save()

    sample = torch.stack(hidden_states, dim=0)  # [num_layers, hidden]
    baseline_samples.append(sample)

baseline_stack = torch.stack(baseline_samples, dim=0)
baseline_mean = baseline_stack.mean(dim=0)
baseline_mean_by_layer = {i: baseline_mean[i] for i in range(num_layers)}

print("baseline_mean:", baseline_mean.shape)
print("layer 0 vector:", baseline_mean_by_layer[0].shape)

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

baseline_mean: torch.Size([80, 8192])
layer 0 vector: torch.Size([8192])


See if we can compute a steering vector for a concept using the previously computed mean baseline vector.

In [6]:
concept = "Dust"

messages = [{"role": "user", "content": f"Tell me about {concept.lower()}"}]
prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# demonstration of how to extract activations from each layer
with model.trace(prompt, remote=True) as tracer:
    hidden_states = []
    for i in range(num_layers):
        hidden_states.append(model.model.layers[i].output[0][-1, :].detach().cpu().save())
    hidden_states.save()

sample = torch.stack(hidden_states, dim=0)  # [num_layers, hidden]

concept_vector_by_layer = {
    idx: sample[idx] - baseline_mean_by_layer[idx] for idx in range(num_layers)
}

⬇ Downloading:   0%|          | 0.00/1.02M [00:00<?]

Sanity check: inject the concept vector during generation on a neutral prompt and verify the output is steered toward the concept. We inject at a middle layer with a moderate multiplier.

In [8]:
inject_layer = num_layers // 2
multiplier = 15.0
steering_vec = concept_vector_by_layer[inject_layer].to(torch.bfloat16)

neutral_prompt = "Tell me about something interesting."
messages = [{"role": "user", "content": neutral_prompt}]
prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# baseline generation (no steering)
with model.generate(prompt, max_new_tokens=100, do_sample=False, remote=True):
    baseline_ids = model.generator.output.save()

baseline_text = model.tokenizer.decode(baseline_ids[0], skip_special_tokens=True)
print("=== Baseline ===")
print(baseline_text)

# steered generation (inject concept vector at every token)
with model.generate(prompt, max_new_tokens=100, do_sample=False, remote=True):
    act = model.model.layers[inject_layer].output[0]
    act[:] += multiplier * steering_vec.to(act.device)
    steered_ids = model.generator.output.save()

steered_text = model.tokenizer.decode(steered_ids[0], skip_special_tokens=True)
print(f"\n=== Steered toward '{concept}' (layer {inject_layer}, {multiplier}x) ===")
print(steered_text)

⬇ Downloading:   0%|          | 0.00/967 [00:00<?]

=== Baseline ===
system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Tell me about something interesting.assistant

I'd like to tell you about the Voynich Manuscript, a mysterious and fascinating book that has been puzzling scholars and cryptographers for centuries.

The Voynich Manuscript is a handwritten book that is estimated to have been created in the 15th century. It is named after the book dealer who purchased it in 1912, Wilfrid Voynich. The manuscript is about 240 pages long and is written in an unknown script and language. The text is accompanied by


⬇ Downloading:   0%|          | 0.00/962 [00:00<?]


=== Steered toward 'Dust' (layer 40, 15.0x) ===
system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

Tell me about something interesting.assistant

dust. It is a major source of allergens and can be a significant problem for people with allergies. However, it also has some interesting properties and plays a crucial role in the ecosystem. For example, it can be used as a food source for some insects and can even be used to help plants grow. Additionally, the study of dust has led to the development of new technologies, such as air purification systems and cleaning products. 
The study of dust is also important for understanding the Earth's climate
